[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_14/TSA_ch14_hedging/TSA_ch14_hedging.ipynb)

# TSA_ch14_hedging

Minimum-variance hedge ratios of the BET with the DAX on daily returns since 2005: DCC, rolling OLS (250 days) and static OLS, estimated on 2005-2014 and evaluated out of sample on 2015-2026 (hedging effectiveness).

*Time Series Analysis (TSA), Chapter 14: Multivariate GARCH models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (step 1: univariate GARCH) is not preinstalled in Google Colab: pip install arch
# (without it, garch11_fit falls back to a numpy maximum-likelihood estimator)
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])
    except Exception as e:
        print('arch not installed, the numpy fallback will be used:', e)

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # default size about 7 inches wide (as in MFM); a chart is shown 11-13 cm wide on a slide, so these font sizes
    # give 7-9 pt text there. Multi-panel charts should stay near 7-7.6 inches wide: at 10-11 inches the axis
    # text drops to about 5 pt on the slide.
    rc['figure.figsize'] = (7.0, 3.4)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)
    _remember_tight_layout()


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
from scipy.signal import lfilter

NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'eurron': 'EUR/RON', 'btc': 'Bitcoin'}

PANEL = ['sp500', 'dax', 'bet', 'eurron', 'btc']

PANEL_START = '2015-01-01'

PAIR_START = '2000-01-01'

HEDGE_START = '2005-01-01'

SPLIT = '2014-12-31'

ROLL = 250

ALPHA_VAR = 0.01

WEIGHTS = (0.5, 0.5)

CRISES = {'2008': ('2008-09-15', '2009-03-31'), '2020': ('2020-02-20', '2020-04-30')}

CALM = ('2017-01-01', '2019-12-31')

PANEL_CRISIS = ('2020-02-14', '2020-06-30')

EX = {'s1': 20.0, 's2': 20.0, 'rhos': (0.2, 0.8), 'a': 0.05, 'b': 0.93, 'qbar': 0.5, 'q11': 1.0, 'q22': 1.0, 'q12': 0.5, 'z': (-2.0, -2.5), 'var_s': (1.5, 1.2), 'var_rho': (0.3, 0.9), 'h_ss': 1.2, 'h_sf': 1.0, 'h_rho': 0.8, 'bk': {'c12': 0.02, 'a11': 0.3, 'a22': 0.25, 'b11': 0.94, 'b22': 0.95, 'e': (-2.0, -1.5), 'h12': 0.5}}

COLORS = {'sp500': st.MainBlue, 'dax': st.Teal, 'bet': st.IDAred, 'eurron': st.Forest, 'btc': st.Amber}

HERE = "."

def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def shade(ax, periods=CRISES, alpha=0.15):
    for i, (a, b) in enumerate(periods.values()):
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=st.Crimson, alpha=alpha, lw=0,
                   label='Crisis periods (2008, 2020)' if i == 0 else None)


def common_returns(names, start, end=None):
    """Daily log returns in % computed on the days on which ALL markets trade: prices are aligned first, so each
    return covers the same calendar interval for every series."""
    P = pd.concat([load_close(k, start) for k in names], axis=1, join='inner')
    if end:
        P = P.loc[:end]
    return (100 * np.log(P).diff()).dropna()


def weekly_returns(names, start=PANEL_START):
    """Weekly log returns in % (Friday closes, or the last close of the week): the weekly horizon removes most of the
    asynchronous-trading problem of markets that close at different hours (New York, Frankfurt, Bucharest, crypto)."""
    P = pd.concat([load_close(k, start).resample('W-FRI').last() for k in names], axis=1).dropna()
    return (100 * np.log(P).diff()).dropna()


def garch11_negloglik(theta, r):
    """Gaussian negative log-likelihood of r_t = mu + e_t, sigma2_t = omega + alpha e_{t-1}^2 + beta sigma2_{t-1}."""
    mu, omega, alpha, beta = theta
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 0.9999:
        return 1e10
    e = r - mu
    s2 = garch11_variance(e, omega, alpha, beta)
    return 0.5 * np.sum(np.log(2 * np.pi) + np.log(s2) + e ** 2 / s2)


def garch11_variance(e, omega, alpha, beta, s0=None):
    """sigma2_t = omega + alpha e_{t-1}^2 + beta sigma2_{t-1}, started at the sample variance (one-step-ahead:
    sigma2_t uses only e_1, ..., e_{t-1})."""
    x = np.empty(len(e))
    x[0] = np.var(e) if s0 is None else s0
    x[1:] = omega + alpha * e[:-1] ** 2
    # sigma2_t - beta sigma2_{t-1} = x_t  -> a linear filter
    out = lfilter([1.0], [1.0, -beta], x)
    return out


def garch11_fit(r):
    """Step 1 of DCC: GARCH(1,1) with a constant mean, Gaussian QML. The arch package if installed, else a numpy
    maximum-likelihood fallback (the two give practically the same estimates)."""
    r = np.asarray(r, float)
    try:
        from arch import arch_model
        res = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='normal').fit(disp='off')
        p = res.params
        return dict(mu=float(p['mu']), omega=float(p['omega']), alpha=float(p['alpha[1]']), beta=float(p['beta[1]']))
    except ImportError:
        v = np.var(r)
        best = optimize.minimize(garch11_negloglik, [r.mean(), 0.05 * v, 0.08, 0.90], args=(r,), method='Nelder-Mead',
                                 options=dict(maxiter=4000, xatol=1e-8, fatol=1e-8))
        mu, omega, alpha, beta = best.x
        return dict(mu=float(mu), omega=float(omega), alpha=float(alpha), beta=float(beta))


def garch11_filter(r, p, s0=None):
    """Conditional volatility (one step ahead) and standardised residuals z_t = (r_t - mu) / sigma_t for given
    parameters p (from garch11_fit), on any sample (also out of sample)."""
    e = np.asarray(r, float) - p['mu']
    s2 = garch11_variance(e, p['omega'], p['alpha'], p['beta'], s0)
    s = np.sqrt(s2)
    return s, e / s


def step1(R, fit_until=None):
    """GARCH(1,1) for every column of R (estimated on R.loc[:fit_until]); volatilities and residuals on all of R."""
    P, S, Z = {}, {}, {}
    for c in R.columns:
        x = R[c].loc[:fit_until] if fit_until else R[c]
        P[c] = garch11_fit(x.values)
        s0 = float(np.var(x.values - P[c]['mu']))
        S[c], Z[c] = garch11_filter(R[c].values, P[c], s0)
    return P, pd.DataFrame(S, index=R.index), pd.DataFrame(Z, index=R.index)


def dcc_path(Z, a, b, Qbar=None):
    """Correlation matrices R_t (T x N x N), one step ahead (R_t uses z_1, ..., z_{t-1}). Every element of Q_t follows
    the same linear recursion, computed with scipy's lfilter."""
    X = np.asarray(Z, float)
    T, N = X.shape
    Qbar = np.corrcoef(X.T) if Qbar is None else Qbar
    P = np.einsum('ti,tj->tij', X, X)                       # z_t z_t'
    drive = np.empty_like(P)
    drive[0] = Qbar
    drive[1:] = (1 - a - b) * Qbar + a * P[:-1]
    Q = lfilter([1.0], [1.0, -b], drive.reshape(T, -1), axis=0).reshape(T, N, N)
    Q[0] = Qbar
    d = np.sqrt(np.einsum('tii->ti', Q))
    return Q / (d[:, :, None] * d[:, None, :])


def dcc_loglik(Z, a, b, Qbar=None):
    """Correlation part of the Gaussian log-likelihood: -1/2 sum_t (log|R_t| + z_t' R_t^{-1} z_t - z_t' z_t)."""
    X = np.asarray(Z, float)
    R = dcc_path(X, a, b, Qbar)
    _, logdet = np.linalg.slogdet(R)
    quad = np.einsum('ti,tij,tj->t', X, np.linalg.inv(R), X)
    return float(-0.5 * np.sum(logdet + quad - np.sum(X ** 2, axis=1)))


def num_hessian(f, x, h=1e-4):
    k = len(x)
    H = np.zeros((k, k))
    E = np.eye(k) * h
    for i in range(k):
        for j in range(k):
            H[i, j] = (f(x + E[i] + E[j]) - f(x + E[i] - E[j]) - f(x - E[i] + E[j]) + f(x - E[i] - E[j])) / (4 * h * h)
    return H


def dcc_fit(Z):
    """Step 2: (a, b) by maximum likelihood with Qbar fixed at the sample correlation of z (correlation targeting).
    Standard errors from the step-2 Hessian (they ignore the estimation error of step 1). CCC: a = b = 0."""
    X = np.asarray(Z, float)
    Qbar = np.corrcoef(X.T)

    def nll(p):
        a, b = p
        if a < 0 or b < 0 or a + b >= 0.9999:
            return 1e10
        return -dcc_loglik(X, a, b, Qbar)

    best = None
    for s in ((0.03, 0.95), (0.01, 0.98), (0.05, 0.90), (0.10, 0.80)):
        r = optimize.minimize(nll, s, method='Nelder-Mead', options=dict(xatol=1e-7, fatol=1e-7, maxiter=4000))
        if best is None or r.fun < best.fun:
            best = r
    a, b = best.x
    try:
        se = np.sqrt(np.diag(np.linalg.inv(num_hessian(nll, best.x))))
    except Exception:
        se = np.array([np.nan, np.nan])
    ll, ll0 = -best.fun, dcc_loglik(X, 0.0, 0.0, Qbar)
    return dict(a=float(a), b=float(b), se_a=float(se[0]), se_b=float(se[1]), loglik=float(ll), ccc_loglik=float(ll0),
                lr=float(2 * (ll - ll0)), Qbar=Qbar)


def half_life(p):
    return float(np.log(0.5) / np.log(p))


def hedge():
    R = common_returns(['bet', 'dax'], HEDGE_START)
    P, S, Z = step1(R, fit_until=SPLIT)
    f = dcc_fit(Z.loc[:SPLIT].values)
    Rt = dcc_path(Z.values, f['a'], f['b'], f['Qbar'])
    h_dcc = pd.Series(Rt[:, 0, 1] * S['bet'] / S['dax'], index=R.index)          # cov / var = rho sigma_s / sigma_f
    ins = R.loc[:SPLIT]
    h_sta = float(np.cov(ins['bet'], ins['dax'])[0, 1] / ins['dax'].var())
    h_rol = (R['bet'].rolling(ROLL).cov(R['dax']) / R['dax'].rolling(ROLL).var()).shift(1)   # past data only
    oos = R.loc[R.index > SPLIT]
    hd, hr = h_dcc.loc[oos.index], h_rol.loc[oos.index]
    un = oos['bet']
    out = {'a': f['a'], 'b': f['b'], 'static': h_sta, 'n_oos': len(oos), 'first': str(oos.index[0].date()),
           'dcc_min': float(hd.min()), 'dcc_max': float(hd.max()), 'dcc_mean': float(hd.mean()),
           'dcc_dmax': str(hd.idxmax().date()), 'dcc_dmin': str(hd.idxmin().date()), 'var_un': float(un.var())}
    for k, h in (('static', h_sta), ('rolling', hr), ('dcc', hd)):
        e = un - h * oos['dax']
        out[f'var_{k}'] = float(e.var())
        out[f'he_{k}'] = float(1 - e.var() / un.var())
    return R, h_dcc, h_rol, h_sta, out


def fig_hedge(save_it=True):
    R, h_dcc, h_rol, h_sta, out = hedge()
    a = '2015-01-01'
    fig, ax = plt.subplots(figsize=(10.17, 2.93))
    ax.plot(h_dcc.loc[a:].index, h_dcc.loc[a:], color=st.IDAred, lw=0.7, label='DCC hedge ratio')
    ax.plot(h_rol.loc[a:].index, h_rol.loc[a:], color=st.MainBlue, lw=1.4, label='Rolling OLS (250 days)')
    ax.axhline(h_sta, color=st.Forest, lw=1.5, ls='--', label='Static OLS (2005-2014)')
    ax.set_ylabel('Units of DAX sold per unit of BET')
    st.legend_outside_bottom(ax, ncol=3)
    save('tsa_ch14_hedge', save_it)
    return out

## Run

In [ ]:
print(fig_hedge())

![tsa_ch14_hedge](./tsa_ch14_hedge.png)

Output: `tsa_ch14_hedge.pdf`